# Check the cosine-sum identity behind Abel's counterexample

*Adversarial Mathematics From First Principles*, [chapter 3](https://programmer.ie/books/math/03-chapter/) · [worked example](https://programmer.ie/examples/math/03-chapter/)

Run the cells in order. Everything below the program is taken from the laboratory's recorded, hash-verified run; the program is a small, self-contained demonstration of the same mathematics.

Cauchy's 1821 textbook claimed that a convergent sum of continuous functions is
continuous; Abel's 1826 counterexample is a series of sines whose sum jumps.
The classical proof of Abel's example begins with a finite identity:

cos t + cos 2t + … + cos Nt = sin((N + 1/2) t) / (2 sin(t/2)) − 1/2, whenever sin(t/2) ≠ 0.

This program checks that first step numerically across a grid of N and t.

## Run it

Standard library only; nothing to install.

In [1]:
"""The finite cosine-sum identity behind Abel's counterexample (standard library only).

    cos t + cos 2t + ... + cos Nt  =  sin((N + 1/2) t) / (2 sin(t/2))  -  1/2,
    for every N >= 0 and every real t with sin(t/2) != 0.

The proof is the Lean theorem; this is an independent numerical check.
"""
import math


def worst_difference(constant):
    ts = [k / 7 for k in range(1, 44)] + [-k / 7 for k in range(1, 44)]
    worst, points = 0.0, 0
    for t in ts:
        if abs(math.sin(t / 2)) < 1e-12:
            continue
        for big_n in range(61):
            lhs = math.fsum(math.cos(n * t) for n in range(1, big_n + 1))
            rhs = math.sin((big_n + 0.5) * t) / (2 * math.sin(t / 2)) - constant
            worst = max(worst, abs(lhs - rhs))
            points += 1
    return worst, points


worst, points = worst_difference(0.5)
print(f"points checked (N = 0..60, t on a grid in (-6.2, 6.2)): {points}")
print(f"largest |left - right| with the constant 1/2: {worst:.1e}")
print(f"N = 0 is the empty sum: left = 0, right = {math.sin(0.5 * 1.0) / (2 * math.sin(0.5)) - 0.5:.1e} at t = 1")
print()
# Failure control: the wrong constant must show up immediately.
worst, _ = worst_difference(1 / 3)
print(f"control: with the constant 1/3 the largest difference is {worst:.4f}")

points checked (N = 0..60, t on a grid in (-6.2, 6.2)): 5246
largest |left - right| with the constant 1/2: 1.7e-13
N = 0 is the empty sum: left = 0, right = 0.0e+00 at t = 1

control: with the constant 1/3 the largest difference is 0.1667


## What the output shows

The two sides agree to rounding error at every point, and with the wrong
constant they disagree by exactly the difference. The identity is the
*finite* building block only: a numerical check at finitely many points is not
a proof, which is why the laboratory proves it in Lean, and neither the
program nor the Lean theorem says anything yet about the infinite series.

## What the laboratory checked

The recorded run `20261113T133159Z-2e4aa81c` ([evidence](https://github.com/ernanhughes/math/tree/main/experiments/investigations/n2-abel-f1/kernel-runs/2026-11-13-pinned)) is the book's result; the program above is a reader-sized demonstration of the same mathematics. **Evidence:** Fresh kernel check (Lean with Mathlib, one kernel) plus numerical computation. **Status:** `kernel_checked_correspondence_pending`.

> For every natural number N and real t with sin(t/2) ≠ 0: cos t + cos 2t + … + cos Nt = sin((N + 1/2) t) / (2 sin(t/2)) − 1/2.

| Check | System | Recorded outcome |
| --- | --- | --- |
| `lean` | Lean kernel | 2 of 2 declarations accepted and bound to their pinned statements |
| `compute` | python | exit 0, verdict identity_holds |

Failure controls: **5 of 5** rejected or refused as required (`lean-wrong-coefficient`, `lean-wrong-lower-bound`, `lean-wrong-upper-bound`, `lean-binding-wrong-statement`, `compute-wrong-constant`).
Correspondence `N2-F1-CORR-01` (does the formal statement say what the claim means?): **pending the author's approval**.
Contract `N2-ABEL-F1` version 1, sha256 `7f9de3e565b397c5…`; `result.json` sha256 `9309d4ac2c86d241…`.

## Not shown

As the laboratory's own record states:

- Anything about the infinite series: Steps 2–5 (integration, the oscillating-integral estimate, convergence, the discontinuity) are not formalized.
- Anything historical: Abel did not state this identity; it belongs to the modern reconstruction.
- A Rocq check: this stage is single-kernel.

## Reproduce or challenge it

To reproduce the laboratory's run, build the `math-env` image from a clone of the repository ([Getting Started](/books/math/29-chapter/)) and run:

```bash
img=$(docker image inspect math-env --format '{{.Id}}')
docker run --rm --network none -e UV_OFFLINE=1 -e MATHENV_IMAGE_ID=$img -v "${PWD}:/source:ro" -v "${PWD}/out:/out" math-env uv run --offline math-engine proof abel-f1 --out /out/proof-abel-f1
```

This Docker form was tested for `fermat`, `kempe` and `simpson`, not separately for `abel-f1`. To re-verify the recorded run itself without Docker, `math-engine proof --verify experiments/investigations/n2-abel-f1/kernel-runs/2026-11-13-pinned` (R-V02).

Examine the finite identity and its domain condition sin(t/2) ≠ 0. Is the
condition exactly right? Does the Lean statement say the same thing as the
English one, including the empty sum at N = 0? Then say which later step of the
proof of Abel's counterexample you would formalize next, and what could go
wrong in its statement.